# Preparation worldclim data¶

In [ ]:
# Libraries
import os, time, shutil, rioxarray
import numpy as np
import xarray as xr
from scipy import ndimage
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir_data = '../data/'
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'

---

### Pre-processing

In [ ]:
# Initialize dask
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(queue='compute', 
                       cores=24, #6
                       memory='256 GB',
                       account='bm0891', 
                       interface="ib0",
                       walltime='01:00:00',
                       local_directory='../dask/',
                       job_extra_directives=[
                           '-o ../dask/LOG_worker_%j.o',
                           '-e ../dask/LOG_worker_%j.o'])

# Scale dask clusters
cluster.scale(jobs=10)

# Configurate dashboard url
import dask
dask.config.config.get('distributed').get('dashboard') \
    .update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Create client
from dask.distributed import Client
client = Client(cluster)

client

In [ ]:
# Function to trim memory of workers
def trim_memory() -> int:
    import ctypes
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

In [ ]:
def prep_worldclim():
    
    """Prepare worldclim data"""

    # Read elevation data
    da_elev = rioxarray.open_rasterio(
        f'{dir_data}worldclim/wc2.1_30s_elev/wc2.1_30s_elev.tif', 
        chunks=dict(y=5000, x=5000))
    
    # Select land cells and export
    da_elev.where(da_elev != da_elev.attrs['_FillValue']) \
        .rename('worldclim_elev') \
        .drop_vars('spatial_ref') \
        .sel(y=slice(80, -60)) \
        .chunk(dict(y=5000, x=5000)) \
        .rio.to_raster(f'{dir01}worldclim_elev.tif')   
    
    # List of bioclimate files
    bio_list = sorted(os.listdir(f'{dir_data}worldclim/wc2.1_30s_bio/'))
    
    for i in bio_list:
        # File Path of bioclimatic files
        file_path_biox = f'{dir_data}worldclim/wc2.1_30s_bio/{i}'
        # File name
        f_bio_x_str = ('worldclim_bio' + 
                       file_path_biox.split('_')[-1].split('.')[0])
        # Read data
        da = rioxarray.open_rasterio(file_path_biox, 
                                     chunks=dict(y=5000, x=5000))       
        # Select land cells and export
        da.where(da != da.attrs['_FillValue']) \
            .rename(f_bio_x_str) \
            .drop_vars('spatial_ref') \
            .sel(y=slice(80, -60)) \
            .chunk(dict(y=5000, x=5000)) \
            .rio.to_raster(dir01 + f_bio_x_str + '.tif') 
        # Release memory
        client.run(trim_memory)


%time prep_worldclim()

In [ ]:
# Close dask cluster
cluster.close()

In [ ]:
# Wait for dask client to completely disconnect
time.sleep(60)

---

### Fill land nans 0.01°

In [ ]:
def fill_nans(da_var, var, dir_out):
    """
    Fills NaN values in the specified variable's dataset using the nearest valid 
    data, applies a land mask, and exports the result to a new Zarr dataset.

    Args:
        var (str): Name of the variable to process (e.g., 'temperature', 'precipitation').
        dir_out (str): Directory where prepared data is stored and the filled dataset will be exported.

    Returns:
        None
    """

    def fill_nans_array(data, invalid):
        """
        Replace invalid (NaN) data cells by the value of the nearest valid data 
        cell.
        """
        ind = ndimage.distance_transform_edt(invalid,
                                             return_distances=False,
                                             return_indices=True)
        return data[tuple(ind)]

    # Paths for input and output
    land_mask_path = os.path.join(dir_out, 'ds_prep_copernicus_land_mask_r001.zarr')
    output_path = os.path.join(dir_out, f'ds_prep_{var}_r001.zarr')

    # Read land mask data
    da_land = xr.open_zarr(land_mask_path) \
                .chunk(dict(lat=5000, lon=5000)) \
                .copernicus_land_mask \
                .compute()

    # Fill nan using function fill_nans_array
    # If there are no NaNs, skip filling process
    if not da_var.isnull().any():
        da_fill = da_var.values  # No filling required
    else:
        da_fill = fill_nans_array(da_var.values, da_var.isnull().values)

    # Create a new Dataset with filled data
    ds_filled = xr.Dataset(dict(lat = da_var.lat, lon=da_var.lon))
    ds_filled[var] = (('lat', 'lon'), da_fill)

    # Apply land mask to the filled data
    ds_filled = ds_filled.where(da_land)

    # Export the filled dataset to Zarr format
    ds_filled.chunk(dict(lat=5000, lon=5000)) \
             .to_zarr(output_path, mode='w')


for i in ['worldclim_bio1', 'worldclim_bio2', 'worldclim_bio3', 
          'worldclim_bio4', 'worldclim_bio5', 'worldclim_bio6',
          'worldclim_bio7', 'worldclim_bio8', 'worldclim_bio9',
          'worldclim_bio10', 'worldclim_bio11', 'worldclim_bio12',
          'worldclim_bio13', 'worldclim_bio14', 'worldclim_bio15',
          'worldclim_bio16', 'worldclim_bio17', 'worldclim_bio18',
          'worldclim_bio19', 'worldclim_elev']:
    # Read data
    da_var = rioxarray.open_rasterio(dir01 + i + '.tif')[0] \
        .rename(dict(y='lat', x='lon'))
    # Fill nans with nearest neighbour if there are nans
    %time fill_nans(da_var, i, dir01)

---

### Post-processing 0.01°

In [ ]:
# Merge files
xr.open_mfdataset(f'{dir01}ds_prep_worldclim_*_r001.zarr', engine='zarr') \
    .drop_vars(['band', 'spatial_ref']) \
    .to_zarr(f'{dir01}ds_prep_worldclim_r001.zarr', mode='w');

---

### Regridding 0.25°

In [ ]:
# Import regridding function
from regrid_high_res_v1_01 import regrid_high_res, prep_tif

In [ ]:
def regrid_da_r025(f_source, dir_target, dir_source, dir_out, 
              size_tiles, fill_value=None, olap=1):  
    """Regrid large xarray dataarrays.

    Args:
        f_source (str): The filename (without extension) of the source .tif file to be regridded.
        dir_target (str): Directory containing target grid .tif file.
        dir_source (str): Directory containing the the source  .tif file.
        dir_out (str): Directory to store the output and intermediate files.
        size_tiles (int): Size of the regridding tiles in degrees.
        fill_value (float, optional): Fill value to use in the regridding process. Defaults to None.
        olap (int, optional): Overlap size in degrees for regridding tiles. Defaults to 1.
        
    Returns:
        xarray.Dataset: The combined dataset after regridding.
    """
    # Prepare the target and source data arrays from TIFF files
    da_target = prep_tif(dir_target + 'target_grid_r025.tif', 'target_grid_r025')
    da_source = prep_tif(dir_source + f_source + '.tif', f_source)
    # Regridd source array to target grid
    regrid_high_res(da_target, da_source, dir_out,
                    account='bm0891', partition='compute',
                    size_tiles=size_tiles, olap=olap, fill_value = fill_value,
                    type_export='zarr', del_interm=False)

In [ ]:
for i in ['worldclim_bio1', 'worldclim_bio2', 'worldclim_bio3', 
          'worldclim_bio4', 'worldclim_bio5', 'worldclim_bio6',
          'worldclim_bio7', 'worldclim_bio8', 'worldclim_bio9',
          'worldclim_bio10', 'worldclim_bio11', 'worldclim_bio12',
          'worldclim_bio13', 'worldclim_bio14', 'worldclim_bio15',
          'worldclim_bio16', 'worldclim_bio17', 'worldclim_bio18',
          'worldclim_bio19', 'worldclim_elev']:
    %time regrid_da_r025(i, dir01, dir01, dir01, 20, -32768, 0.1)
#Takes about 3-4 min for one file

---

### Fill nans 0.025°

In [ ]:
# Fill nans of 0.25° files with nearest-neighbour
def fill_nans_r025(var, dir_out):
    """
    Fills NaN values in the specified variable's dataset using the nearest valid 
    data, applies a land mask, and exports the result to a new Zarr dataset.

    Args:
        var (str): Name of the variable to process (e.g., 'temperature', 'precipitation').
        dir_out (str): Directory where prepared data is stored and the filled dataset will be exported.

    Returns:
        None
    """

    def fill_nans_array(data, invalid):
        """
        Replace invalid (NaN) data cells by the value of the nearest valid data 
        cell.
        """
        ind = ndimage.distance_transform_edt(invalid,
                                             return_distances=False,
                                             return_indices=True)
        return data[tuple(ind)]

    # Paths for input and output
    land_mask_path = os.path.join(dir_out, 'ds_luh2_land_mask_r025.nc')
    var_data_path = os.path.join(dir_out, f'ds_regridded_{var}.zarr')
    output_path = os.path.join(dir_out, f'ds_prep_{var}_r025.zarr')

    # Read land mask data
    da_land = xr.open_dataset(land_mask_path) \
                .chunk(dict(lat=5000, lon=5000)) \
                .luh2_land_mask \
                .compute()

    # Read variable data
    da_var = xr.open_zarr(var_data_path)['regridded_' + var]

    # Fill nan using function fill_nans_array
    # If there are no NaNs, skip filling process
    if not da_var.isnull().any():
        da_fill = da_var.values  # No filling required
    else:
        da_fill = fill_nans_array(da_var.values, da_var.isnull().values)

    # Create a new Dataset with filled data
    ds_filled = xr.Dataset(dict(lat = da_var.lat, lon=da_var.lon))
    ds_filled[var] = (('lat', 'lon'), da_fill)

    # Apply land mask to the filled data
    ds_filled = ds_filled.where(da_land)

    # Export the filled dataset to Zarr format
    ds_filled.chunk(dict(lat=5000, lon=5000)) \
             .to_zarr(output_path, mode='w')


for i in ['worldclim_bio1', 'worldclim_bio2', 'worldclim_bio3', 
          'worldclim_bio4', 'worldclim_bio5', 'worldclim_bio6',
          'worldclim_bio7', 'worldclim_bio8', 'worldclim_bio9',
          'worldclim_bio10', 'worldclim_bio11', 'worldclim_bio12',
          'worldclim_bio13', 'worldclim_bio14', 'worldclim_bio15',
          'worldclim_bio16', 'worldclim_bio17', 'worldclim_bio18',
          'worldclim_bio19', 'worldclim_elev']:
    %time fill_nans_r025(i, dir01)

---

### Post-processing 0.25°

In [ ]:
# Merge files
xr.open_mfdataset(f'{dir01}ds_prep_worldclim_*_r025.zarr', engine='zarr') \
    .to_zarr(f'{dir01}ds_prep_worldclim_r025.zarr', mode='w');

In [ ]:
for i in ['worldclim_bio1', 'worldclim_bio2', 'worldclim_bio3', 
          'worldclim_bio4', 'worldclim_bio5', 'worldclim_bio6',
          'worldclim_bio7', 'worldclim_bio8', 'worldclim_bio9',
          'worldclim_bio10', 'worldclim_bio11', 'worldclim_bio12',
          'worldclim_bio13', 'worldclim_bio14', 'worldclim_bio15',
          'worldclim_bio16', 'worldclim_bio17', 'worldclim_bio18',
          'worldclim_bio19', 'worldclim_elev']:
    shutil.rmtree(f'{dir01}ds_prep_{i}_r025.zarr')